# Homework 5 — Two exploration rates

Individual. Due after Session 10.

You are given a short DQN training loop and a base-stock return. Run epsilon 0.05 and epsilon 0.50. Fill `NOTE`: did either beat a fair reading of the base-stock daily pay, and would you deploy either run?


In [ ]:
import numpy as np

CAPACITY = 4
GAMMA = 0.95
HOLD, MARGIN, STOCKOUT = 0.2, 2.0, 1.5
NOTE = ""  # TODO: at least 80 characters. Deploy or not, and why.


def step(stock, order, rng):
    demand = int(rng.integers(0, 3))
    started = min(CAPACITY, stock + order)
    sales = min(started, demand)
    ending = started - sales
    missed = max(0, demand - started)
    pay = MARGIN * sales - HOLD * ending - STOCKOUT * missed
    return ending, pay, demand


class TinyQ:
    def __init__(self, rng):
        self.w1 = rng.normal(0, 0.2, size=(2, 8))
        self.b1 = np.zeros(8)
        self.w2 = rng.normal(0, 0.2, size=(8, 3))
        self.b2 = np.zeros(3)

    def forward(self, x):
        self.x = np.asarray(x, dtype=float)
        self.z1 = self.x @ self.w1 + self.b1
        self.h = np.maximum(0, self.z1)
        return self.h @ self.w2 + self.b2

    def learn(self, x, action, target, lr=0.05):
        q = self.forward(x)
        grad_q = np.zeros_like(q)
        grad_q[action] = np.clip(q[action] - target, -1.0, 1.0)
        self.w2 -= lr * np.outer(self.h, grad_q)
        self.b2 -= lr * grad_q


def base_stock_daily(days=300):
    rng = np.random.default_rng(11)
    stock, total = 1, 0.0
    for _ in range(days):
        order = max(0, min(2, 2 - stock))
        stock, pay, _demand = step(stock, order, rng)
        total += pay
    return total / days


def train(epsilon, steps=800):
    rng = np.random.default_rng(3)
    net = TinyQ(np.random.default_rng(3))
    memory = []
    stock, demand = 1, 1
    recent = []
    for _ in range(steps):
        x = np.array([stock / CAPACITY, demand / 2.0])
        q = net.forward(x)
        action = int(rng.integers(0, 3)) if rng.random() < epsilon else int(np.argmax(q))
        ending, pay, new_demand = step(stock, action, rng)
        nxt = np.array([ending / CAPACITY, new_demand / 2.0])
        memory.append((x, action, pay, nxt))
        bx, ba, br, bn = memory[int(rng.integers(0, len(memory)))]
        net.learn(bx, ba, br + GAMMA * np.max(net.forward(bn)))
        recent.append(pay)
        stock, demand = ending, new_demand
    return float(np.mean(recent[-100:]))


print("base-stock average daily pay", round(base_stock_daily(), 3))
for epsilon in (0.05, 0.50):
    print(f"epsilon {epsilon}: last-100 average pay", round(train(epsilon), 3))
print("PASS" if len(NOTE) >= 80 else "NOT YET write NOTE (deploy or not, and why)")
